# Outline
1. download yolo model
2. run yolo predictions on each page of the dataset
3. run ocr and map it into yolo nodes

In [3]:
from huggingface_hub import hf_hub_download

filepath = hf_hub_download(
    repo_id="juliozhao/DocLayout-YOLO-DocLayNet-Docsynth300K_pretrained",
    filename="doclayout_yolo_doclaynet_imgsz1120_docsynth_pretrain.pt",
    local_dir="models_weights",
    local_dir_use_symlinks=False
)

print(f"Done! Model is saved: {filepath}")

/Users/zrutix/Projects/Bakalarka/gat-document-relation/.venv/lib/python3.12/site-packages/huggingface_hub/utils/_validators.py:205: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Done! Model is saved: models_weights/doclayout_yolo_doclaynet_imgsz1120_docsynth_pretrain.pt


In [17]:
from doclayout_yolo import YOLOv10
from tqdm import tqdm
import json, os
from PIL import Image

PNG_PATH   = "data/DocLayNet/PNG"
MODEL_PATH = "models_weights/doclayout_yolo_doclaynet_imgsz1120_docsynth_pretrain.pt"
DATA_FOLDERS = [
    "data/DocLayNet/train_data",
    "data/DocLayNet/val_data",
    "data/DocLayNet/test_data",
]

model = YOLOv10(MODEL_PATH)
print("Model loaded")

def run_yolo_on_graph(pil_image, model):
    page_width, page_height = pil_image.size

    # Perform prediction
    yolo_results = model.predict(
        source=pil_image,
        imgsz=1120,
        conf=0.2,
        iou=0.5,
        agnostic_nms=True,
        device="mps",
        verbose=False
    )
    
    yolo_result = yolo_results[0]

    valid_boxes = []
    for box in yolo_result.boxes:
        x1, y1, x2, y2 = box.xyxy[0].tolist()
        category_id = int(box.cls[0].item())
        class_name  = yolo_result.names[category_id]
        confidence  = box.conf[0].item()
        valid_boxes.append({
            "coords":     [x1, y1, x2, y2],
            "class_id":   category_id,
            "class_name": class_name,
            "confidence": confidence
        })

    # IoA filtering — keeping larger element
    boxes_to_remove = set()
    threshold = 0.80

    for i in range(len(valid_boxes)):
        if i in boxes_to_remove: continue
        for j in range(i + 1, len(valid_boxes)):
            if j in boxes_to_remove: continue

            box_i = valid_boxes[i]["coords"]
            box_j = valid_boxes[j]["coords"]

            x_left   = max(box_i[0], box_j[0])
            y_top    = max(box_i[1], box_j[1])
            x_right  = min(box_i[2], box_j[2])
            y_bottom = min(box_i[3], box_j[3])

            if x_right < x_left or y_bottom < y_top:
                continue

            intersection_area = (x_right - x_left) * (y_bottom - y_top)
            area_i = (box_i[2] - box_i[0]) * (box_i[3] - box_i[1])
            area_j = (box_j[2] - box_j[0]) * (box_j[3] - box_j[1])

            ioa_i = intersection_area / area_i if area_i > 0 else 0
            ioa_j = intersection_area / area_j if area_j > 0 else 0

            if ioa_i > threshold and ioa_j > threshold:
                if valid_boxes[i]["confidence"] > valid_boxes[j]["confidence"]:
                    boxes_to_remove.add(j)
                else:
                    boxes_to_remove.add(i)
                    break
            elif ioa_i > threshold:
                boxes_to_remove.add(i)
                break
            elif ioa_j > threshold:
                boxes_to_remove.add(j)

    filtered_boxes = [b for i, b in enumerate(valid_boxes) if i not in boxes_to_remove]

    # Build yolo_nodes
    yolo_nodes = []
    for node_id, box_data in enumerate(filtered_boxes):
        x1, y1, x2, y2 = box_data["coords"]

        norm_x1 = x1 / page_width
        norm_y1 = y1 / page_height
        norm_x2 = x2 / page_width
        norm_y2 = y2 / page_height

        center_x = (norm_x1 + norm_x2) / 2
        center_y = (norm_y1 + norm_y2) / 2
        width    = norm_x2 - norm_x1
        height   = norm_y2 - norm_y1

        yolo_nodes.append({
            "node_id":          node_id,
            "label":            box_data["class_name"],
            "label_id":         box_data["class_id"],
            "yolo_confidence":  box_data["confidence"],
            "text":             "",
            "geometry": {
                "absolute_pixel_coords": [int(x1), int(y1), int(x2), int(y2)],
                "normalized_coords":     [norm_x1, norm_y1, norm_x2, norm_y2],
                "normalized_center":     [center_x, center_y],
                "normalized_size":       [width, height]
            }
        })

    return yolo_nodes

for folder in DATA_FOLDERS:
    graph_files = [f for f in os.listdir(folder) if f.endswith(".json")]
    updated = 0
    skipped = 0

    for filename in tqdm(graph_files, desc=folder.split("/")[-1]):
        graph_path = os.path.join(folder, filename)
        with open(graph_path, encoding="utf-8") as f:
            graph = json.load(f)

        if "yolo_nodes" in graph:
            skipped += 1
            continue

        img_path = os.path.join(PNG_PATH, graph["file_name"])  # ✅ fix
        if not os.path.exists(img_path):
            skipped += 1
            continue

        pil_image = Image.open(img_path).convert("RGB")
        graph["yolo_nodes"] = run_yolo_on_graph(pil_image, model)

        with open(graph_path, "w", encoding="utf-8") as f:
            json.dump(graph, f)
        updated += 1

    print(f"  {folder.split('/')[-1]}: {updated} updated | {skipped} skipped")

print("\nDone!")

Model loaded


train_data: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 19106/19106 [37:06<00:00,  8.58it/s]


  train_data: 19106 updated | 0 skipped


val_data: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 9239/9239 [17:45<00:00,  8.67it/s]


  val_data: 9239 updated | 0 skipped


test_data: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 6895/6895 [13:10<00:00,  8.72it/s]

  test_data: 6895 updated | 0 skipped

Done!


In [4]:
import os
import json
import random
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.image as mpimg

PNG_PATH   = "data/DocLayNet/PNG"
DATA_FOLDERS = {
    "train": "data/DocLayNet/train_data",
    "val":   "data/DocLayNet/val_data",
    "test":  "data/DocLayNet/test_data",
}
OUT_DIR    = "visual_pages"
PER_FOLDER = 100

os.makedirs(OUT_DIR, exist_ok=True)

for split_name, folder in DATA_FOLDERS.items():
    split_out = os.path.join(OUT_DIR, split_name)
    os.makedirs(split_out, exist_ok=True)

    all_files = [f for f in os.listdir(folder) if f.endswith(".json")]

    # Skip pages without yolo nodes
    with_yolo    = [f for f in all_files if "yolo_nodes" in json.load(open(os.path.join(folder, f)))]
    without_yolo = [f for f in all_files if f not in with_yolo]

    sample = with_yolo[:PER_FOLDER]
    if len(sample) < PER_FOLDER:
        sample += random.sample(without_yolo, min(PER_FOLDER - len(sample), len(without_yolo)))

    print(f"{split_name}: {len(sample)} stránok ({len(with_yolo)} má yolo_nodes)")

    for filename in sample:
        graph_path = os.path.join(folder, filename)
        with open(graph_path, encoding="utf-8") as f:
            graph = json.load(f)

        W, H = graph["width"], graph["height"]

        # Load image
        img_path = os.path.join(PNG_PATH, graph.get("file_name", ""))
        img = mpimg.imread(img_path) if os.path.exists(img_path) else None

        fig, axes = plt.subplots(1, 2, figsize=(20, 10))
        fig.suptitle(filename, fontsize=10, y=1.01)

        panels = [
            ("Ground Truth", graph.get("nodes", []),      "lime",  True),
            ("YOLO Predictions", graph.get("yolo_nodes", []), "red", False),
        ]

        for ax, (title, nodes, color, use_xywh) in zip(axes, panels):
            ax.set_title(title, fontsize=13)
            ax.set_xlim(0, W)
            ax.set_ylim(H, 0)

            if img is not None:
                ax.imshow(img, extent=[0, W, H, 0], aspect='auto', alpha=0.9)

            for node in nodes:
                if use_xywh:
                    x, y, w, h = node["bbox"]
                    label = f"{node['category_name']} ({node['node_id']})"
                else:
                    x1, y1, x2, y2 = node["geometry"]["absolute_pixel_coords"]
                    x, y, w, h = x1, y1, x2 - x1, y2 - y1
                    label = f"{node['label']} ({node['node_id']})"

                ax.add_patch(patches.Rectangle(
                    (x, y), w, h, linewidth=1.5,
                    edgecolor=color, facecolor=color, alpha=0.15
                ))
                ax.add_patch(patches.Rectangle(
                    (x, y), w, h, linewidth=1.5,
                    edgecolor=color, facecolor="none"
                ))
                ax.text(x + 2, y - 4, label, fontsize=6, color=color,
                        bbox=dict(facecolor="black", alpha=0.45, pad=1, edgecolor="none"))

            ax.set_aspect("equal")
            ax.axis("off")

        plt.tight_layout()
        out_path = os.path.join(split_out, filename.replace(".json", ".png"))
        plt.savefig(out_path, dpi=100, bbox_inches="tight")
        plt.close()

    print(f"  → Saved to {split_out}/")

print("\Done!")

train: 100 stránok (19106 má yolo_nodes)
  → uložené do visual_pages/train/
val: 100 stránok (9239 má yolo_nodes)
  → uložené do visual_pages/val/
test: 100 stránok (6895 má yolo_nodes)


KeyboardInterrupt: 

In [ ]:
import os
import json
import numpy as np
import easyocr
from PIL import Image
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

# Configuration
PNG_PATH     = "data/DocLayNet/PNG"
DATA_FOLDERS = [
    "data/DocLayNet/train_data",
    "data/DocLayNet/val_data",
    "data/DocLayNet/test_data",
]
MARGIN = 10

# EasyOCR reader
reader = easyocr.Reader(['en'], gpu=True)
print("EasyOCR loaded")


def load_graph(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)

def save_graph(path, graph):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(graph, f)

for folder in DATA_FOLDERS:
    graph_files = sorted([f for f in os.listdir(folder) if f.endswith(".json")])
    paths       = [os.path.join(folder, f) for f in graph_files]

    with ThreadPoolExecutor(max_workers=8) as pool:
        graphs = list(tqdm(pool.map(load_graph, paths), total=len(paths),
                           desc=f"Loading {folder.split('/')[-1]}"))

    updated  = 0
    skipped  = 0
    no_image = 0

    for graph_path, graph in tqdm(zip(paths, graphs), total=len(paths),
                                   desc=folder.split("/")[-1]):

        # skip if yolo_nodes is missing
        if "yolo_nodes" not in graph:
            skipped += 1
            continue

        # Resume — skip if text is already extracted
        if all(node.get("text", "") != "" for node in graph["yolo_nodes"]):
            skipped += 1
            continue

        img_path = os.path.join(PNG_PATH, graph["file_name"])
        if not os.path.exists(img_path):
            no_image += 1
            continue

        # Full-page OCR
        page_image_np = np.array(Image.open(img_path).convert("RGB"))
        ocr_results   = reader.readtext(page_image_np)

        # add text to yolo nodes
        for node in graph["yolo_nodes"]:
            x1, y1, x2, y2 = node["geometry"]["absolute_pixel_coords"]
            node_texts = []

            if ocr_results and ocr_results[0] is not None:
                for (ocr_box, text, confidence) in ocr_results:
                    ocr_center_x = sum(pt[0] for pt in ocr_box) / 4.0
                    ocr_center_y = sum(pt[1] for pt in ocr_box) / 4.0

                    if (x1 - MARGIN <= ocr_center_x <= x2 + MARGIN) and \
                       (y1 - MARGIN <= ocr_center_y <= y2 + MARGIN):
                        node_texts.append(text)

            node["text"] = " ".join(node_texts).strip()

        save_graph(graph_path, graph)
        updated += 1

    print(f"  {folder.split('/')[-1]}: {updated} updated | {skipped} skipped | {no_image} missing image")

print("\nDone!")

EasyOCR loaded


train_data:   0%|                                                                                                                                                                               | 0/19106 [00:00<?, ?it/s]/Users/zrutix/Projects/Bakalarka/gat-document-relation/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
